In [1]:
import pandas as pd
from pathlib import Path

dfs = []
for i in range(1999, 2025):
    path = f"StateDrugUtilizationData{i}.csv"
    if Path(path).exists():
        dfs.append(pd.read_csv(path))
    else:
        print(f"Missing: {path}")

combined_df = pd.concat(dfs, ignore_index=True)

/var/folders/vs/b2d0657s1v5g396b2w0fwgqw0000gn/T/ipykernel_62926/3600261723.py:8: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(path))
/var/folders/vs/b2d0657s1v5g396b2w0fwgqw0000gn/T/ipykernel_62926/3600261723.py:8: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(path))
/var/folders/vs/b2d0657s1v5g396b2w0fwgqw0000gn/T/ipykernel_62926/3600261723.py:8: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(path))
/var/folders/vs/b2d0657s1v5g396b2w0fwgqw0000gn/T/ipykernel_62926/3600261723.py:8: DtypeWarning: Columns (2,5) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(path))
/var/folders/vs/b2d0657s1v5g396b2w0fwgqw0000gn/T/ipykernel_62926/3600261723.py:8: DtypeWarning: Columns (2,4,5) have mixed types. Specify dtyp

In [ ]:
len(combined_df)

In [3]:
combined_df = combined_df.sort_values('Product Name')

In [4]:
combined_df.tail(30)

,Utilization Type,State,NDC,Labeler Code,Product Code,Package Size,Year,Quarter,Suppression Used,Product Name,Units Reimbursed,Number of Prescriptions,Total Amount Reimbursed,Medicaid Amount Reimbursed,Non Medicaid Amount Reimbursed
1237213,FFSU,MT,186045131,186,451,31,1999,1,True,NaN,NaN,NaN,NaN,NaN,NaN
1237214,FFSU,MT,186045158,186,451,58,1999,1,False,NaN,575.0,17.0,567.01,0.0,0.0
1237215,FFSU,MT,186045231,186,452,31,1999,1,True,NaN,NaN,NaN,NaN,NaN,NaN
1237216,FFSU,MT,186045258,186,452,58,1999,1,True,NaN,NaN,NaN,NaN,NaN,NaN
1237217,FFSU,MT,186060631,186,606,31,1999,1,True,NaN,NaN,NaN,NaN,NaN,NaN
1237219,FFSU,MT,186074231,186,742,31,1999,1,False,NaN,5033.0,160.0,18208.37,0.0,0.0
1237220,FFSU,MT,186074282,186,742,82,1999,1,True,NaN,NaN,NaN,NaN,NaN,NaN
1237221,FFSU,MT,186074331,186,743,31,1999,1,True,NaN,NaN,NaN,NaN,NaN,NaN
1237249,FFSU,MT,187320426,187,3204,26,1999,1,True,NaN,NaN,NaN,NaN,NaN,NaN
1239139,FFSU,MT,597005205,597,52,5,1999,1,True,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
print(combined_df.shape)
print(combined_df['Labeler Code'].dtype)
print(combined_df[combined_df['Labeler Code'] == '00074']['Product Name'].unique())

(94966348, 15)
int64
[]


In [12]:
lupron_rows = combined_df[
    (combined_df['Labeler Code'] == 74) &
    (combined_df['Product Name'].str.strip().str.contains('LUPRON', na=False))
]

print(lupron_rows['Product Name'].unique())
print(lupron_rows.shape)

['LUPRON DEP']
(41638, 15)


In [14]:
# Filter to LUPRON DEPOT only
lupron_df = combined_df[
    (combined_df['Labeler Code'] == 74) &
    (combined_df['Product Name'].str.strip().str.contains('LUPRON', na=False))
].copy()

# Check what you have
print(lupron_df['Year'].unique())
print(lupron_df['State'].unique())
print(lupron_df['Suppression Used'].value_counts())

[2020 2010 2018 2017 2015 2024 2012 2019 2016 2022 2021 2023 2013 2011
 2009 2014]
['ID' 'AK' 'PA' 'MA' 'FL' 'TX' 'MN' 'NV' 'GA' 'CA' 'MI' 'OK' 'SC' 'ND'
 'XX' 'NJ' 'CT' 'NE' 'NH' 'AZ' 'MS' 'AL' 'NY' 'VA' 'WV' 'KS' 'WA' 'WY'
 'ME' 'DC' 'NC' 'SD' 'MO' 'IA' 'MD' 'AR' 'WI' 'OH' 'PR' 'NM' 'DE' 'RI'
 'MT' 'VT' 'LA' 'KY' 'IL' 'TN' 'IN' 'HI' 'CO' 'OR' 'UT']
Suppression Used
True     22499
False    19139
Name: count, dtype: int64


In [16]:
# Drop unknown state and suppressed rows
lupron_clean = lupron_df[
    (lupron_df['Suppression Used'] == False) &
    (~lupron_df['State'].isin(['XX']))
].copy()

# Aggregate to state-year
lupron_agg = (
    lupron_clean
    .groupby(['State', 'Year'])['Number of Prescriptions']
    .sum()
    .reset_index()
)

# Check result
print(lupron_agg.shape)
print(lupron_agg.head(10))
print(lupron_agg['Year'].nunique(), 'years')
print(lupron_agg['State'].nunique(), 'states')

(770, 3)
  State  Year  Number of Prescriptions
0    AK  2009                     13.0
1    AK  2010                     27.0
2    AK  2011                     11.0
3    AK  2012                     44.0
4    AK  2013                     40.0
5    AK  2014                     14.0
6    AK  2015                     11.0
7    AK  2016                     20.0
8    AK  2017                     76.0
9    AK  2018                     70.0
16 years
52 states


In [20]:
!pip install pymc arviz numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.8/562.8 kB 10.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 36.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.

In [ ]:
!pip install "numpy<2.0" pymc arviz

  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
INFO: pip is looking at multiple versions of pytensor to determine which version is compatible with other requirements. This could take a while.
  Using cached pandas-3.0.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached pandas-3.0.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pandas-2.3.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pandas-2.3.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pandas-2.3.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pandas-2.2.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (89 kB)
INFO: pip is still looking at multiple versions of pytensor to determine which version is compatible with other requirements. This could take a while.
  Using cached pandas-2.2.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (

In [ ]:
import pymc as pm
import arviz as az
import numpy as np

# Encode state as integer index
lupron_agg['state_idx'] = pd.Categorical(lupron_agg['State']).codes
n_states = lupron_agg['state_idx'].nunique()

# Center year
lupron_agg['year_centered'] = lupron_agg['Year'] - lupron_agg['Year'].mean()

# Arrays for model
state_idx = lupron_agg['state_idx'].values
year_c = lupron_agg['year_centered'].values
y = lupron_agg['Number of Prescriptions'].values

with pm.Model() as hierarchical_model:

    # --- Hyperpriors (national level) ---
    mu_alpha = pm.Normal('mu_alpha', mu=0, sigma=500)
    sigma_alpha = pm.HalfNormal('sigma_alpha', sigma=200)

    mu_beta = pm.Normal('mu_beta', mu=0, sigma=50)
    sigma_beta = pm.HalfNormal('sigma_beta', sigma=25)

    # --- State-level parameters (partial pooling) ---
    alpha = pm.Normal('alpha', mu=mu_alpha, sigma=sigma_alpha, shape=n_states)
    beta = pm.Normal('beta', mu=mu_beta, sigma=sigma_beta, shape=n_states)

    # --- Expected value ---
    mu = alpha[state_idx] + beta[state_idx] * year_c

    # --- Likelihood ---
    sigma_obs = pm.HalfNormal('sigma_obs', sigma=200)
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma_obs, observed=y)

    # --- Sample ---
    trace = pm.sample(2000, tune=1000, target_accept=0.9, 
                      return_inferencedata=True, random_seed=42)